# Manufacturing Defect Analysis: One-Way ANOVA

This notebook contains the statistical analysis supporting the **Manufacturing Defect Analysis** portfolio case study.

A prior Pareto analysis identified Production Models **595130**, **595214**, and **595242** as the three highest-defect production models, with total defect counts of **888**, **663**, and **630**, respectively.

The purpose of this notebook is to evaluate whether mean defect-category frequency differs significantly among these three production models.

## Original Data Workflow

The original analysis used CSV files exported from the Power BI analysis environment:

- `Top_Production_Model_Defects_Project2.csv` — used to verify the three highest-defect production models
- `Model1_Defects.csv` — Production Model 595130
- `Model2_Defects.csv` — Production Model 595214
- `Model3_Defects.csv` — Production Model 595242

The model-specific files contain defect categories and their corresponding frequencies. Those frequencies were then used as the numerical observations in the ANOVA.

Because the original course-provided CSV files are not redistributed in the public portfolio repository, this notebook first looks for the original files in `../data/`. If they are unavailable, it uses an embedded summary of the same model-level defect frequencies solely to reproduce the statistical calculations.

## Analytical Question and Hypotheses

**Question:** Is there a statistically significant difference in mean defect-category frequency among Production Models 595130, 595214, and 595242?

- **Null hypothesis (H₀):** Mean defect-category frequency is equal among the three production models.
- **Alternative hypothesis (Hₐ):** At least one production model has a mean defect-category frequency that differs from the others.
- **Significance level:** α = 0.05

In [1]:
from pathlib import Path

import pandas as pd
from scipy.stats import f_oneway

DATA_DIR = Path("../data")

expected_files = {
    "overall": DATA_DIR / "Top_Production_Model_Defects_Project2.csv",
    "model_595130": DATA_DIR / "Model1_Defects.csv",
    "model_595214": DATA_DIR / "Model2_Defects.csv",
    "model_595242": DATA_DIR / "Model3_Defects.csv",
}

all_files_available = all(path.exists() for path in expected_files.values())
print("Original CSV files available:", all_files_available)

Original CSV files available: False


## Load and Verify the Data

In [2]:
if all_files_available:
    overall = pd.read_csv(expected_files["overall"])
    model_595130 = pd.read_csv(expected_files["model_595130"])
    model_595214 = pd.read_csv(expected_files["model_595214"])
    model_595242 = pd.read_csv(expected_files["model_595242"])

    print("Loaded original CSV files from ../data/")
    display(overall.head())
else:
    print(
        "The original course-provided CSV files are not included in the public repository. "
        "Using the aggregated defect frequencies from the original analysis to reproduce the ANOVA."
    )

    model_595130 = pd.DataFrame({
        "Defect Type": [
            "missing component", "missing/unknown", "no property assembled",
            "component misalignment", "bad modules", "wrong configuration",
            "reversed component", "insufficient solder", "damaged/ lifted pad",
            "long terminals", "lifted component", "component broken",
            "wrong component", "pin damaged", "wrong routing"
        ],
        "Count of Defect Type": [181, 172, 171, 160, 32, 30, 28, 19, 18, 17, 16, 15, 15, 7, 7]
    })

    model_595214 = pd.DataFrame({
        "Defect Type": [
            "solder bridge", "component height/titled", "excessive solder",
            "reversed component", "insufficient solder", "pin hole",
            "wrong component", "bad modules", "damaged/ lifted pad",
            "trace open", "component misalignment", "no property assembled",
            "wrong configuration", "defective component", "component broken",
            "wrong routing", "pin damaged"
        ],
        "Count of Defect Type": [136, 133, 128, 116, 22, 21, 20, 18, 18, 14, 9, 9, 8, 5, 3, 2, 1]
    })

    model_595242 = pd.DataFrame({
        "Defect Type": [
            "solder bridge", "lifted component", "pin hole", "component broken",
            "pin damaged", "damaged/ lifted pad", "insufficient solder",
            "damaged component", "long terminals", "wrong component",
            "defective component", "missing/unknown", "trace open",
            "wrong configuration", "missing component", "bad modules"
        ],
        "Count of Defect Type": [150, 144, 140, 100, 13, 12, 12, 11, 11, 8, 7, 7, 5, 5, 3, 2]
    })

The original course-provided CSV files are not included in the public repository. Using the aggregated defect frequencies from the original analysis to reproduce the ANOVA.


### Verify Model Totals

The three model-specific files should reproduce the defect totals identified by the earlier Pareto analysis.

In [3]:
summary = pd.DataFrame({
    "Production Model": ["595130", "595214", "595242"],
    "Total Defects": [
        model_595130["Count of Defect Type"].sum(),
        model_595214["Count of Defect Type"].sum(),
        model_595242["Count of Defect Type"].sum()
    ],
    "Number of Defect Categories": [
        len(model_595130),
        len(model_595214),
        len(model_595242)
    ],
    "Mean Defect Frequency": [
        model_595130["Count of Defect Type"].mean(),
        model_595214["Count of Defect Type"].mean(),
        model_595242["Count of Defect Type"].mean()
    ]
})

summary.round(2)

,Production Model,Total Defects,Number of Defect Categories,Mean Defect Frequency
0,595130,888,15,59.20
1,595214,663,17,39.00
2,595242,630,16,39.38


The totals confirm **888**, **663**, and **630** defects for Models 595130, 595214, and 595242, respectively.

## Prepare the ANOVA Inputs

The individual defect-category frequencies provide multiple numerical observations for each production-model group.

In [4]:
freq_595130 = model_595130["Count of Defect Type"].to_numpy()
freq_595214 = model_595214["Count of Defect Type"].to_numpy()
freq_595242 = model_595242["Count of Defect Type"].to_numpy()

print("Model 595130 frequencies:", freq_595130)
print("Model 595214 frequencies:", freq_595214)
print("Model 595242 frequencies:", freq_595242)

Model 595130 frequencies: [181 172 171 160  32  30  28  19  18  17  16  15  15   7   7]
Model 595214 frequencies: [136 133 128 116  22  21  20  18  18  14   9   9   8   5   3   2   1]
Model 595242 frequencies: [150 144 140 100  13  12  12  11  11   8   7   7   5   5   3   2]


## One-Way ANOVA

In [5]:
f_statistic, p_value = f_oneway(freq_595130, freq_595214, freq_595242)

print(f"F-statistic: {f_statistic:.3f}")
print(f"P-value: {p_value:.3f}")
print("Significance level: 0.05")

F-statistic: 0.579
P-value: 0.565
Significance level: 0.05


### Interpretation

The one-way ANOVA produces an **F-statistic of 0.579** and a **p-value of 0.565**. Because the p-value is greater than the 0.05 significance level, the null hypothesis is **not rejected**.

The analysis does not provide sufficient evidence that mean defect-category frequency differs significantly among Production Models 595130, 595214, and 595242.

This does **not** prove that the three models are identical. Model 595130 still has the highest total defect count, but the statistical test does not establish that its mean defect-category frequency is significantly different from those of the other two models.

## Exploratory Pairwise Comparisons

The original analysis also compared each pair of production models as exploratory follow-up checks.

In [6]:
comparisons = [
    ("595130 vs 595214", freq_595130, freq_595214),
    ("595130 vs 595242", freq_595130, freq_595242),
    ("595214 vs 595242", freq_595214, freq_595242),
]

pairwise_results = []

for label, group_a, group_b in comparisons:
    f_value, p_val = f_oneway(group_a, group_b)
    pairwise_results.append({
        "Comparison": label,
        "F-statistic": f_value,
        "P-value": p_val
    })

pd.DataFrame(pairwise_results).round(3)

,Comparison,F-statistic,P-value
0,595130 vs 595214,0.873,0.358
1,595130 vs 595242,0.747,0.394
2,595214 vs 595242,0.000,0.984


All three exploratory pairwise p-values exceed 0.05, which is consistent with the overall ANOVA result.

## Business Interpretation

The statistical results do not support restricting corrective action to a single production model. A broader process-improvement review is appropriate while continuing to monitor model-specific defect patterns.

Model 595130 should still receive added attention because it has the highest total defect count. However, the ANOVA result should be interpreted alongside the Pareto and root-cause analyses, which show different defect concentrations across the three models and identify potential shared process conditions for further investigation.

## Limitations

- The ANOVA evaluates differences in **mean defect-category frequency**; it does not identify the causes of defects.
- Failure to reject the null hypothesis does not demonstrate that the three models are identical.
- Root-cause conclusions require additional process observations, equipment checks, procedure reviews, and inspection audits.
- The original course-provided CSV files are not included in the public repository because no explicit redistribution license was provided.
- When those files are unavailable, the embedded model-level frequency summaries are used only to reproduce the statistical calculations from the original analysis.